# PPY - Class 05: Generator `send()`, Coroutines & Advanced OOP
**Topics:** `yield` as expression, `send()`, coroutine pattern, advanced class exercises

## 🧠 Theory

### `yield` vs `return`
```python
# return ends the function; yield pauses and saves state

def gen():
    yield 1   # pause here
    yield 2   # resume, pause here
```

### Using `send()`
```python
def echo():
    while True:
        received = (yield)          # yield as expression
        print(f"Got: {received}")

g = echo()
next(g)          # MUST prime the generator first
g.send(10)       # prints "Got: 10"
g.send("hello")  # prints "Got: hello"
```

Key rules:
- First call must be `next(gen)` or `gen.send(None)`
- `send(value)` resumes generator; `yield` evaluates to `value`
- Generator can both yield a value AND receive one:
  `x = yield result`

### Iterator internals
```python
it = iter(lst)
while True:
    try:    x = next(it)
    except StopIteration: break
```

## ✏️ Task 1 – filter_even_send()

In [1]:
def filter_even_send():
    while True:
        n = (yield)
        if n is not None and n % 2 == 0:
            yield n
        else:
            yield None

gen = filter_even_send()
next(gen)
for n in [1, 2, 3, 4, 5, 6]:
    result = gen.send(n)
    print(f"send({n}) -> {result}")

send(1) -> None
send(2) -> None
send(3) -> None
send(4) -> None
send(5) -> None
send(6) -> None


## ✏️ Task 2 – cumulative_product_send()

In [ ]:
def cumulative_product_send():
    product = 1
    while True:
        n = (yield product)
        if n is not None:
            product *= n

gen = cumulative_product_send()
next(gen)
for n in [1, 2, 3, 4]:
    print(f"send({n}) -> {gen.send(n)}")

## ✏️ Task 3 – random_walk_send()

In [ ]:
def random_walk_send(start=0):
    pos = start
    while True:
        step = (yield pos)
        if step is not None:
            pos += step

gen = random_walk_send()
next(gen)
for step in [1, -1, 1, 1, -1]:
    print(f"send({step:+d}) -> position {gen.send(step)}")

## ✏️ Task 4 – prefixes_send()

In [ ]:
def prefixes_send():
    while True:
        s = (yield)
        if s:
            for i in range(1, len(s) + 1):
                yield s[:i]

gen = prefixes_send()
next(gen)
gen.send("hello")      # prime with first word
for prefix in gen:     # collect remaining prefixes
    print(prefix)
    break if False else None   # generator exhausts naturally after all prefixes

## ✏️ Task 5 – pairs_send()

In [ ]:
def pairs_send():
    buf   = None
    count = 0
    while count < 10:
        val = (yield)
        count += 1
        if buf is None:
            buf = val
            yield None
        else:
            pair = (buf, val)
            buf  = None
            yield pair
    if buf is not None:
        yield (buf,)

gen = pairs_send()
next(gen)
for n in [1, 2, 3, 4, 5, 10]:
    r = gen.send(n)
    print(f"send({n}) -> {r}")

## ✏️ Task 6 – fib_limit_send()

In [ ]:
def fib_limit_send(n):
    a, b = 0, 1
    while a <= n:
        cont = (yield a)
        if cont is False:
            return
        a, b = b, a + b

gen     = fib_limit_send(10)
results = [next(gen)]
try:
    for _ in range(6):
        results.append(gen.send(True))
    gen.send(False)
except StopIteration:
    pass

print("Fibonacci up to 10:", results)

## ✏️ Other exercises – most_frequent, anagram_groups

In [ ]:
def most_frequent(lst):
    counts = {}
    best   = (None, 0)
    for x in lst:
        counts[x] = counts.get(x, 0) + 1
        if counts[x] > best[1]:
            best = (x, counts[x])
    return best

print("Most frequent:", most_frequent(["a","b","a","c","b","a","d"]))

def anagram_groups(words):
    groups = {}
    for w in words:
        key = tuple(sorted(w))
        groups.setdefault(key, []).append(w)
    return {k: v for k, v in groups.items() if len(v) > 1}

words  = ["eat","tea","tan","ate","nat","bat","listen","silent","enlist"]
groups = anagram_groups(words)
print("Anagram groups:", list(groups.values()))
print("Longest group: ", max(groups.values(), key=len))

## ✏️ Generator utility functions (Tasks 6-11)

In [ ]:
from itertools import islice

# unique elements preserving order
def unique_gen(lst):
    seen = set()
    for x in lst:
        if x not in seen:
            seen.add(x); yield x

print("Unique:", list(unique_gen([1,3,2,1,4,3,5,2])))

# words without split()
def words_gen(s):
    word = ""
    for c in s:
        if c == " ":
            if word: yield word; word = ""
        else: word += c
    if word: yield word

print("Words:", list(words_gen("hello world foo bar")))

# cumulative products
def cum_product(lst):
    p = 1
    for x in lst: p *= x; yield p

print("Cum product:", list(cum_product([1,2,3,4])))

# with index (no enumerate)
def with_index(lst):
    i = 0
    for x in lst: yield (i, x); i += 1

print("Indexed:", list(with_index(["a","b","c"])))

# flatten one level
def flatten(nested):
    for sub in nested:
        for x in sub: yield x

print("Flat:", list(flatten([[1,2],[3,4],[5]])))

# alternating
def alternating():
    while True: yield 1; yield -1

print("Alt:", list(islice(alternating(), 8)))

## ✏️ OOP extras – best3 leaderboard

In [ ]:
def best3(stream):
    scores  = {}
    leaders = []
    for name, score in stream:
        scores[name] = max(scores.get(name, 0), score)
        if len(scores) >= 3:
            new_leaders = sorted(scores.items(), key=lambda x: -x[1])[:3]
            if new_leaders != leaders:
                leaders = new_leaders
                print("leaders changed:", leaders)
    return sorted(scores.items(), key=lambda x: -x[1])[:3]

lst = [("a",14),("b",17),("a",15),("b",7),("c",16),("e",10),
       ("f",14),("e",17),("a",13),("c",19),("d",15),("b",18),
       ("d",11),("a",12),("d",10),("d",10)]
print("Final winners:", best3(lst))

## ✏️ OOP extras – compose()

In [ ]:
def compose(*funcs):
    def composed(*args, **kwargs):
        result = funcs[-1](*args, **kwargs)
        for f in reversed(funcs[:-1]):
            if isinstance(result, tuple):   result = f(*result)
            elif isinstance(result, dict):  result = f(**result)
            else:                           result = f(result)
        return result
    return composed

def h(x, **kwargs): return (x + 2, sum(kwargs.values()))
def g(*args):       return {"d": len(args), "s": sum(args)}
def f(**kwargs):    return f"Resulting dict: {kwargs}"

print(compose(f, g, h)(10, a=1, b=2, c=3))

---
## 📋 Cheatsheet - Class 05

### Generator basics
```python
def gen():
    yield 1       # pause, return 1
    yield 2       # resume, return 2

gen_expr = (x**2 for x in range(5))  # lazy
```

### Using send()
```python
def coroutine():
    while True:
        x = (yield)          # receive value
        yield x * 2          # send back result

c = coroutine()
next(c)                      # prime
c.send(5)                    # -> 10
```

### Iterator protocol
```python
it = iter(lst)
while True:
    try:    x = next(it)
    except StopIteration: break
```

### send() rules
- First call MUST be `next(gen)` or `gen.send(None)`
- `send(val)` resumes; `yield` evaluates to `val`
- Generator can yield and receive simultaneously: `x = yield result`
